# Middleware and guardrails

Assistant version 8: Redact PII, pause refunds over $100, add fallback and enforce limits.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra agents --group authoring`, then `uv run --extra agents --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## before_agent, before_model, wrap_model_call, after_model, wrap_tool_call, after_agent; hook ordering

Predict the output before running. The function definitions below match `examples/01_hooks.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Observe all six middleware hook boundaries around an actual tool call.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/01_hooks.py --offline
Expected: Hook names show node hooks and nested model/tool wrappers.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from langchain.agents import create_agent

from common.errors import run_cli
from common.fixtures import tool_request
from common.guardrails import HookLog
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    log = HookLog()
    agent = create_agent(
        get_chat_model(
            settings=settings,
            responses=[tool_request("check_stock", {"sku": "BW-001"}), "Checked."],
        ),
        make_order_tools("C001"),
        middleware=[log],
    )
    # KEY IDEA: wrappers surround a call; node hooks run at agent/model boundaries.
    agent.invoke({"messages": [{"role": "user", "content": "Check BW-001 stock."}]})
    print_text(str(log.events))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Decorator and AgentMiddleware class; input/output guardrails

Predict the output before running. The function definitions below match `examples/02_custom_middleware.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare decorator hooks and a class-based input guard.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/02_custom_middleware.py --offline
Expected: A decorator observes valid input; oversized input is rejected before generation.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import after_model

from common.errors import run_cli
from common.guardrails import InputLimit
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


@after_model
def inspect_output(state: dict, runtime: Any) -> None:
    if len(str(state["messages"][-1].content)) > 2000:
        raise ValueError("Answer exceeds the application review limit.")


def main() -> None:
    settings = lesson_settings()
    agent = create_agent(
        get_chat_model(settings=settings), tools=[], middleware=[InputLimit(), inspect_output]
    )
    # KEY IDEA: prompts ask; deterministic hooks enforce these particular limits.
    print_text(
        agent.invoke({"messages": [{"role": "user", "content": "Explain the return window."}]})[
            "messages"
        ][-1].content
    )
    try:
        agent.invoke({"messages": [{"role": "user", "content": "x" * 4001}]})
    except ValueError as error:
        print_text(str(error))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## PIIMiddleware; email and card-like-number redaction; synthetic fixtures

Predict the output before running. The function definitions below match `examples/03_pii.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Redact synthetic email and card-like data before model access and output.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/03_pii.py --offline
Expected: Redacted markers replace synthetic identifiers in retained messages.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    guards = [
        PIIMiddleware("email", strategy="redact", apply_to_output=True, apply_to_tool_results=True),
        PIIMiddleware(
            "card_fixture",
            detector=r"\b(?:\d[ -]?){15}\d\b",
            strategy="redact",
            apply_to_output=True,
            apply_to_tool_results=True,
        ),
    ]
    # KEY IDEA: this custom detector catches card-like fixtures, not every form of PII.
    agent = create_agent(
        get_chat_model(
            settings=settings, responses=["Contact demo@example.test; fixture 4111 1111 1111 1111."]
        ),
        tools=[],
        middleware=guards,
    )
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Synthetic demo@example.test; fixture 4111 1111 1111 1111.",
                }
            ]
        }
    )
    for message in result["messages"]:
        print_text(str(message.content))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## HumanInTheLoopMiddleware; route refunds over $100 to an approval-only tool; approve/reject/resume

Predict the output before running. The function definitions below match `examples/04_approval.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Pause a high-value refund, then demonstrate separate approve and reject decisions.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/04_approval.py --offline
Expected: Approval writes once; rejection writes nothing.
Cost: Up to four model calls in the scripted path; offline no API fee. Only disposable ledgers
change.
Requires: agents
"""

import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

from langgraph.types import Command

from common.errors import run_cli
from common.fixtures import tool_request
from common.guardrails import guarded_agent
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text
from common.returns import SupportContext, runtime_orders


def main() -> None:
    settings = lesson_settings()
    for decision in ["approve", "reject"]:
        with TemporaryDirectory() as folder:
            context = SupportContext(
                "C001",
                runtime_orders(Path(folder) / "orders.db"),
                inspected_unused=True,
                proof_and_accessories=True,
            )
            agent = guarded_agent(
                settings,
                [
                    tool_request(
                        "approve_refund",
                        {
                            "order_id": "BW1001",
                            "sku": "BW-014",
                            "reason": "unused",
                            "idempotency_key": "approval-demo",
                        },
                    ),
                    "Follow the reviewed tool result; no payment sent.",
                ],
            )
            config = thread_config("C001", decision)
            paused = agent.invoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": "Request return of BW-014 on BW1001 with key approval-demo.",
                        }
                    ]
                },
                config,
                context=context,
            )
            print_text(str(paused.get("__interrupt__")))
            # KEY IDEA: this is a simulated human decision, never an agent-generated approval.
            if paused.get("__interrupt__"):
                agent.invoke(
                    Command(resume={"decisions": [{"type": decision}]}), config, context=context
                )
            with closing(sqlite3.connect(context.database)) as db:
                count = db.execute(
                    "SELECT COUNT(*) FROM returns WHERE idempotency_key='approval-demo'"
                ).fetchone()[0]
            print_text(f"{decision}: new records={count}")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## ModelFallbackMiddleware; primary fails, backup responds

Predict the output before running. The function definitions below match `examples/05_fallback.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Recover from a synthetic primary model failure using model fallback.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/05_fallback.py --offline
Expected: Backup response appears after the injected primary failure.
Cost: One backup generation; primary failure is deliberately injected in every mode.
Requires: agents
"""

from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware, wrap_model_call

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


@wrap_model_call
def inject_primary_failure(request: Any, handler: Any) -> Any:
    if request.model is PRIMARY:
        raise TimeoutError("Synthetic primary outage")
    return handler(request)


def main() -> None:
    global PRIMARY
    settings = lesson_settings()
    PRIMARY = get_chat_model(settings=settings)
    backup = get_chat_model(
        settings=settings, responses=["Backup draft: review Brightwave policy before replying."]
    )
    # KEY IDEA: the outer fallback must wrap the failing model operation.
    agent = create_agent(
        PRIMARY, tools=[], middleware=[ModelFallbackMiddleware(backup), inject_primary_failure]
    )
    print_text(
        agent.invoke({"messages": [{"role": "user", "content": "Draft a support reply."}]})[
            "messages"
        ][-1].content
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## ModelCallLimitMiddleware; ToolCallLimitMiddleware; stop runaway loops

Predict the output before running. The function definitions below match `examples/06_call_limits.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Bound model calls and tool dispatch in a deliberately repeating agent.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/06_call_limits.py --offline
Expected: The run ends with a visible limit message rather than looping.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from itertools import repeat

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    for guard in [
        ModelCallLimitMiddleware(run_limit=2),
        ToolCallLimitMiddleware(run_limit=1, exit_behavior="end"),
    ]:
        model = get_chat_model(
            settings=settings, responses=repeat(tool_request("check_stock", {"sku": "BW-001"}))
        )
        # KEY IDEA: call counts constrain amplification; they do not measure exact spending.
        agent = create_agent(model, make_order_tools("C001"), middleware=[guard])
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "Check BW-001 stock repeatedly."}]},
            {"recursion_limit": 20},
        )
        print_text(str(result["messages"][-1].content))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## ModelRetryMiddleware; ToolRetryMiddleware; avoid repeated write side effects

Predict the output before running. The function definitions below match `examples/07_retry.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Retry one transient model failure and one read-tool failure with no delay.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/07_retry.py --offline
Expected: Three model attempts and two read attempts; no write is retried.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware, wrap_model_call
from langchain.tools import tool

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    attempts = {"model": 0, "tool": 0}

    @wrap_model_call
    def flaky_model(request: Any, handler: Any) -> Any:
        attempts["model"] += 1
        if attempts["model"] == 1:
            raise TimeoutError("Synthetic transient model error")
        return handler(request)

    @tool
    def flaky_stock(sku: str) -> dict:
        """Synthetic read-only stock service fails on its first attempt."""
        attempts["tool"] += 1
        if attempts["tool"] == 1:
            raise TimeoutError("Synthetic transient read error")
        return {"sku": sku, "stock": 87}

    options = {"max_retries": 1, "initial_delay": 0, "jitter": False, "retry_on": (TimeoutError,)}
    # KEY IDEA: retry only selected transient errors and idempotent/read operations.
    agent = create_agent(
        get_chat_model(
            settings=settings,
            responses=[tool_request("flaky_stock", {"sku": "BW-001"}), "Stock checked."],
        ),
        [flaky_stock],
        middleware=[
            ModelRetryMiddleware(**options),
            flaky_model,
            ToolRetryMiddleware(tools=["flaky_stock"], **options),
        ],
    )
    agent.invoke({"messages": [{"role": "user", "content": "Check BW-001 stock."}]})
    print_text(str(attempts))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## LLMToolSelectorMiddleware; tool relevance versus extra model cost

Predict the output before running. The function definitions below match `examples/08_tool_selector.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Use a separate model to select relevant tools from a reviewed set.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/08_tool_selector.py --offline
Expected: Only the reviewed stock tool is selected; selection adds model calls.
Cost: Two agent calls plus up to two selection calls in the scripted path.
Requires: agents
"""

from itertools import repeat

from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    selector = get_chat_model(
        settings=settings,
        responses=repeat(tool_request("ToolSelectionResponse", {"tools": ["check_stock"]})),
    )
    model = get_chat_model(
        settings=settings,
        responses=[tool_request("check_stock", {"sku": "BW-001"}), "Checked stock."],
    )
    # KEY IDEA: relevance selection narrows schemas, but authorization still lives in tools.
    agent = create_agent(
        model,
        make_order_tools("C001"),
        middleware=[LLMToolSelectorMiddleware(model=selector, max_tools=1, max_retries=0)],
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "Check stock for BW-001."}]})
    print_text(str([(m.type, m.content) for m in result["messages"]]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## TodoListMiddleware; inspect task planning state

Predict the output before running. The function definitions below match `examples/09_todo.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Track a small support plan through TodoListMiddleware.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/09_todo.py --offline
Expected: The final state contains two pending tasks; planning does not execute them.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    todos = [
        {"content": "Read return policy", "status": "pending"},
        {"content": "Review customer order", "status": "pending"},
    ]
    # KEY IDEA: writing a plan records intentions; it does not execute or verify those tasks.
    model = get_chat_model(
        settings=settings,
        responses=[
            tool_request("write_todos", {"todos": todos}),
            "Plan recorded; both tasks still need execution.",
        ],
    )
    agent = create_agent(model, tools=[], middleware=[TodoListMiddleware()])
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Plan an order and policy review. Do not execute it yet.",
                }
            ]
        }
    )
    print_text(str(result["todos"]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Dynamic system prompt; runtime model selection; immutable customer scope

Predict the output before running. The function definitions below match `examples/10_dynamic_model.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Change the prompt and selected model without changing authenticated scope.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/examples/10_dynamic_model.py --offline
Expected: A runtime prompt and model override keep the same customer context.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, wrap_model_call

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.returns import SupportContext


@dynamic_prompt
def support_prompt(request: Any) -> str:
    return (
        f"Authenticated Brightwave customer: {request.runtime.context.customer_id}. "
        "Do not change this scope."
    )


def main() -> None:
    settings = lesson_settings()
    base = get_chat_model(settings=settings)
    review = get_chat_model(
        settings=settings,
        responses=["Review this complex request with the customer scope unchanged."],
    )

    @wrap_model_call
    def route(request: Any, handler: Any) -> Any:
        # KEY IDEA: model selection changes computation, not authorization context.
        selected = review if len(str(request.messages[-1].content)) > 80 else base
        return handler(request.override(model=selected))

    agent = create_agent(
        base, tools=[], context_schema=SupportContext, middleware=[support_prompt, route]
    )
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Review this Brightwave return with many exceptions. " * 2,
                }
            ]
        },
        context=SupportContext("C001", Path("unused")),
    )
    print_text(result["messages"][-1].content)
    print_text(
        "Both instances use the configured profile here; routing does not imply a stronger model."
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Redact PII, pause refunds over $100, add fallback and enforce limits

Predict the output before running. The function definitions below match `use_case/guarded_agent.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Run a guarded refund request and leave high-value writes paused for review.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/12_middleware/use_case/guarded_agent.py --offline
Expected: A human-review interrupt appears; the synthetic ledger remains unchanged.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from common.errors import run_cli
from common.fixtures import tool_request
from common.guardrails import guarded_agent
from common.lesson import lesson_settings
from common.memory import thread_config
from common.pretty import print_text
from common.returns import SupportContext, runtime_orders


def main() -> None:
    settings = lesson_settings()
    with TemporaryDirectory() as folder:
        context = SupportContext(
            "C001",
            runtime_orders(Path(folder) / "orders.db"),
            inspected_unused=True,
            proof_and_accessories=True,
        )
        agent = guarded_agent(
            settings,
            [
                tool_request(
                    "approve_refund",
                    {
                        "order_id": "BW1001",
                        "sku": "BW-014",
                        "reason": "unused",
                        "idempotency_key": "guarded-demo",
                    },
                )
            ],
        )
        # KEY IDEA: interruption is a successful safety outcome, not a completed refund.
        state = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": "Please return BW-014 on BW1001; use key guarded-demo.",
                    }
                ]
            },
            thread_config("C001", "review"),
            context=context,
        )
        print_text(str(state.get("__interrupt__", state["messages"][-1].content)))
        print_text("Demo stops here. A real UI must authenticate the reviewer before resuming.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/12_middleware/tests` for the supplied solution checks.